In [1]:
import os
from langgraph.graph import StateGraph, END
from ai_framework.nodes import *
# from ai_framework.nodes import completedProcess,thinking_steps,generate_answer

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def route_after_validation(state: GraphState):
    if state["is_valid"]:
        return "thinking"
    else:
        return "end"

def build_graph():
    builder = StateGraph(GraphState)
    builder.add_node("validate", validate_question)
    builder.add_node("Thinking", thinking_steps)
    builder.add_node("retrive_document", generate_answer)
    builder.add_node("Websearch", websearch)
    builder.add_node("suggest_questions", suggest_questions)
    builder.set_entry_point("validate")
    builder.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "thinking": "Thinking",
            "end":END
        }
    )
    builder.add_edge("Thinking", "retrive_document")
    builder.add_edge("retrive_document", "Websearch")
    builder.add_edge("Websearch", "suggest_questions")
    builder.add_edge("suggest_questions", END)
    return builder.compile()

# builder.add_node("websearch", websearch)
    # builder.add_node("followupquestion", suggest_questions)

In [3]:
# -----------------------------
# STREAM FUNCTION
# -----------------------------
def ask_question_stream(query: str):
    graph = build_graph()
    print("\n=== STREAMING START ===\n")
    final_state = {}
    inputObj={"query": query,
              "filename":"reactaa.pdf",
              "messageId":"1234",
              "user_id":"21a1ff59-f04b-450e-bf46-322617dae796",  #session['user_id'] 
              "user_name":"pranay", #session['user_name'] 
              "filename":'reactaa.pdf'
              }

    for step in graph.stream(inputObj):
        print('step',step)
        for node, output in step.items():
            print("-"*10)
            print("node", node)
            print("output", output)
            print("-"*10)
            final_state.update(output)
    print("\n=== STREAMING END ===\n")
    return final_state


In [4]:
result=ask_question_stream("What is the significance of keys in React?")
result


=== STREAMING START ===

step {'validate': {'is_valid': True}}
----------
node validate
output {'is_valid': True}
----------
step {'Thinking': {'thinking': {'content': '{\n    "Steps": [\n        "Step 1: Understand Context": "First, I will understand the context of the question and identify the relevance of keys in React.",\n        "Step 2: Retrieve Information": "Next, I will retrieve related information from React documentation and other reliable sources using RAG to gather details about keys.",\n        "Step 3: Analyze and Summarize": "Then, I will analyze the retrieved information and summarize the significance of keys in React, including', 'used_tokens': 100, 'prompt_tokens': 158, 'total_tokens': 258, 'messageid': '1234'}}}
----------
node Thinking
output {'thinking': {'content': '{\n    "Steps": [\n        "Step 1: Understand Context": "First, I will understand the context of the question and identify the relevance of keys in React.",\n        "Step 2: Retrieve Information": 

{'is_valid': True,
 'thinking': {'content': '{\n    "Steps": [\n        "Step 1: Understand Context": "First, I will understand the context of the question and identify the relevance of keys in React.",\n        "Step 2: Retrieve Information": "Next, I will retrieve related information from React documentation and other reliable sources using RAG to gather details about keys.",\n        "Step 3: Analyze and Summarize": "Then, I will analyze the retrieved information and summarize the significance of keys in React, including',
  'used_tokens': 100,
  'prompt_tokens': 158,
  'total_tokens': 258,
  'messageid': '1234'},
 'documentAnswer': {'content': 'The significance of keys in React is that they are used to uniquely identify and differentiate between components [reactaa.pdf, pg 2]. They help React identify which items have changed, added, or removed. Additionally, keys are crucial when dealing with elements in a list, as they help React efficiently update the DOM when the list changes [

In [5]:
#prompt = template.replace("{query}",state["query"])
#print(prompt) 